## Agrégation des différents indices.
### Structure générale

1. Découpage des couches en entrée par les limites du canton de Vaud
2. Masquage des couches en entrée par les limites de la couche de sol (proxy terres arables/naturelles)
3. Normalisation pour produits finaux
4. Création d'indices combinés

### 1. Découpage des couches en entrée par les limites du canton de Vaud

In [ ]:
# Découpage des GeoTIFF des différents indices par le vecteur du canton de Vaud afin d'avoir une donnée homogène.
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import os

def align_to_reference(
    input_raster,
    reference_raster,
    output_raster,
    resampling=Resampling.bilinear,
    nodata_value=-9999.0
):
    """
    Reproject, resample and mask input_raster on reference_raster grid.
    Handles missing src_crs by assuming reference CRS.
    """

    with rasterio.open(reference_raster) as ref:
        ref_profile = ref.profile
        ref_data = ref.read(1)
        ref_crs = ref.crs
        ref_nodata = ref.nodata

    with rasterio.open(input_raster) as src:
        src_data = src.read(1)

        # --- CORRECTION CRS MANQUANT ---
        src_crs = src.crs if src.crs is not None else ref_crs

        dst_data = np.full(
            (ref_profile["height"], ref_profile["width"]),
            nodata_value,
            dtype=np.float32
        )

        reproject(
            source=src_data,
            destination=dst_data,
            src_transform=src.transform,
            src_crs=src_crs,
            dst_transform=ref_profile["transform"],
            dst_crs=ref_crs,
            resampling=resampling,
            src_nodata=src.nodata,
            dst_nodata=nodata_value
        )

    # Masque agricole (NoData du raster sol)
    dst_data[ref_data == ref_nodata] = nodata_value

    ref_profile.update(
        dtype="float32",
        nodata=nodata_value,
        compress="LZW"
    )

    os.makedirs(os.path.dirname(output_raster), exist_ok=True)

    with rasterio.open(output_raster, "w", **ref_profile) as dst:
        dst.write(dst_data, 1)


### 2. Masquage des couches en entrée par les limites de la couche de sol (proxy terres arables/naturelles)

In [ ]:
# Découpage cartes climatologiques généraux selon le raster sol
from pathlib import Path

reference = "../data/processed/Sol_index_normalized.tif"
input_dir = "../data/processed/Climat/indices_secheresse_ensemble50-50/"
output_dir = "../data/processed/Indices finaux/Climat50-50/"

Path(output_dir).mkdir(parents=True, exist_ok=True)

for tif in Path(input_dir).glob("*.tif"):
    out_name = tif.stem + "_clip" + tif.suffix
    out = Path(output_dir) / out_name

    align_to_reference(
        input_raster=str(tif),
        reference_raster=reference,
        output_raster=str(out)
    )


In [ ]:
# Découpages cartes climat param selon le raster sol
from pathlib import Path

reference = "../data/processed/Sol_index_normalized.tif"
input_dir = "../data/processed/Climat/param_secheresse_ensemble/"
output_dir = "../data/processed/Indices finaux/Climat_param/"

Path(output_dir).mkdir(parents=True, exist_ok=True)

for tif in Path(input_dir).glob("*.tif"):
    out_name = tif.stem + "_clip" + tif.suffix
    out = Path(output_dir) / out_name

    align_to_reference(
        input_raster=str(tif),
        reference_raster=reference,
        output_raster=str(out)
    )

In [ ]:
# Découpage cartes topo selon le raster sol
reference = "../data/processed/Sol_index_normalized.tif"

input_topo = "../data/processed/Topo_index_50m.tif"
output_topo = "../data/processed/Indices finaux/Topo/Topo_index_50m_clip.tif"

align_to_reference(
    input_raster=input_topo,
    reference_raster=reference,
    output_raster=output_topo
)

In [ ]:
# Découpage carte erosion selon raster sol
reference = "../data/processed/Sol_index_normalized.tif"

input_topo = "../app/data/erosion-quantitativ_2056.tif"
output_topo = "../app/data/erosion-quantitativ_2056_clip.tif"

align_to_reference(
    input_raster=input_topo,
    reference_raster=reference,
    output_raster=output_topo
)

### 3. Normalisation pour produits finaux

In [ ]:
# Normalisation pour produits finaux
import os
import json
import numpy as np
import rasterio
from rasterio.enums import Resampling

def normalize_with_clip(input_path, output_path, *,
                        pmin=2.0, pmax=98.0,
                        prelog=False,
                        symmetric=False,
                        nodata_keep=True,
                        nodata_value=None,
                        overwrite=False):
    """
    Normalise un raster avec clipping percentile et min-max.
    - symmetric=True : pour variables centrées sur 0 (ex: courbure). On calcule le percentile pmax
      de la distribution des valeurs absolues puis on clip à +/- that_value.
    - prelog=True : applique np.log1p avant clipping (utile pour TWI)
    - pmin,pmax : percentiles (0..100) ; if symmetric only pmax used.
    - nodata_keep : conserve nodata (remplacé par np.nan pendant calcul).
    """
    if os.path.exists(output_path) and not overwrite:
        raise FileExistsError(f"{output_path} exists. Use overwrite=True to replace.")

    with rasterio.open(input_path) as src:
        profile = src.profile.copy()
        arr = src.read(1).astype(np.float32)
        src_nodata = src.nodata if nodata_value is None else nodata_value

    # mask nodata
    if src_nodata is not None:
        mask = (arr == src_nodata) | (~np.isfinite(arr))
    else:
        mask = ~np.isfinite(arr)
    valid = arr[~mask]

    if valid.size == 0:
        raise ValueError("No valid pixels found in input raster.")

    # optionally apply pre-log (on valid values only)
    if prelog:
        valid_proc = np.log1p(valid)
    else:
        valid_proc = valid

    # symmetric clipping around 0 for centered variables
    if symmetric:
        # use percentile on absolute values
        abs_vals = np.abs(valid_proc)
        cutoff = np.percentile(abs_vals, pmax)
        low_val = -cutoff
        high_val = cutoff
        print(f"[symmetric] cutoff (abs) @ p{pmax} = {cutoff:.6g}")
    else:
        low_val = np.percentile(valid_proc, pmin)
        high_val = np.percentile(valid_proc, pmax)
        print(f"percentiles: p{pmin}={low_val:.6g}, p{pmax}={high_val:.6g}")

    # Create working copy (float) and replace nodata with nan
    work = arr.astype(np.float32)
    work[mask] = np.nan

    # if prelog, transform entire working array's valid pixels
    if prelog:
        with np.errstate(invalid='ignore'):
            work = np.where(np.isfinite(work), np.log1p(work), work)

    # clip
    work_clipped = np.clip(work, low_val, high_val)

    # min-max normalize
    denom = (high_val - low_val)
    if denom == 0:
        # constant field after clipping
        norm = np.full_like(work_clipped, np.nan, dtype=np.float32)
        norm[np.isfinite(work_clipped)] = 0.5
    else:
        norm = (work_clipped - low_val) / denom
    norm[mask] = profile.get("nodata", np.nan) if nodata_keep else np.nan

    # update profile
    profile.update(dtype=rasterio.float32, compress='deflate')
    if nodata_keep:
        profile.update(nodata=profile.get("nodata", -9999))
        out_nodata = profile["nodata"]
    else:
        out_nodata = np.nan

    # ensure output dir
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # write raster
    with rasterio.open(output_path, "w", **profile) as dst:
        write_arr = np.where(np.isfinite(norm), norm, out_nodata).astype(np.float32)
        dst.write(write_arr, 1)

    # save metadata about normalization
    meta = {
        "input": os.path.basename(input_path),
        "output": os.path.basename(output_path),
        "pmin": pmin,
        "pmax": pmax,
        "prelog": bool(prelog),
        "symmetric": bool(symmetric),
        "low_val": float(low_val),
        "high_val": float(high_val),
        "nodata_kept": bool(nodata_keep),
    }
    meta_path = os.path.splitext(output_path)[0] + "_meta.json"
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"Saved normalized raster: {output_path}")
    print(f"Saved metadata: {meta_path}")
    return output_path, meta_path

In [ ]:
# Normalisation des indices climat finaux
from pathlib import Path

input_dir = "../data/processed/Indices finaux/Climat50-50/"
output_dir = "../data/processed/Indices finaux/Climat_norm50-50/"

Path(output_dir).mkdir(parents=True, exist_ok=True)

for tif in Path(input_dir).glob("*.tif"):
    out_name = tif.stem + "_norm" + tif.suffix
    out = Path(output_dir) / out_name

    normalize_with_clip(
        input_path=str(tif),
        output_path=str(out),
        pmin=2.0,
        pmax=98.0,
        prelog=False,
        symmetric=False,
        nodata_keep=True,
        overwrite=True
    )

In [ ]:
#Normalisation des indices topo finaux
from pathlib import Path

input_dir = "../data/processed/Indices finaux/Topo/"
output_dir = "../data/processed/Indices finaux/Topo_norm/"

Path(output_dir).mkdir(parents=True, exist_ok=True)

for tif in Path(input_dir).glob("*.tif"):
    out_name = tif.stem + "_norm" + tif.suffix
    out = Path(output_dir) / out_name

    normalize_with_clip(
        input_path=str(tif),
        output_path=str(out),
        pmin=2.0,
        pmax=98.0,
        prelog=False,
        symmetric=False,
        nodata_keep=True,
        overwrite=True
    )

### 4. Création d'indices combinés

In [ ]:
# Calcul des indices combinés
# On commence par les indices absolus (normalisés entre 0 et 1)
import rasterio
import numpy as np

# --- Chemins vers tes rasters normalisés ---
climat_path = "../data/processed/Indices finaux/Climat_norm/I_climat_ref91_ensemble_clip_norm.tif"
sol_path = "../data/processed/Indices finaux/Sol_norm/Sol_index_normalized.tif"
topo_path = "../data/processed/Indices finaux/Topo_norm/Topo_index_50m_clip_norm.tif"


# --- Poids ---
weights = {
    "climat": 0.33,
    "sol": 0.33,
    "topo": 0.34
}

# --- Lecture des rasters ---
with rasterio.open(climat_path) as src:
    climat = src.read(1).astype('float32')
    mask = src.read_masks(1) == 0  # True où il y a no-data
    profile = src.profile

with rasterio.open(sol_path) as src:
    sol = src.read(1).astype('float32')

with rasterio.open(topo_path) as src:
    topo = src.read(1).astype('float32')

# --- Appliquer le masque de no-data à toutes les couches ---
climat[mask] = np.nan
sol[mask] = np.nan
topo[mask] = np.nan

# --- Calcul de l'indice composite ---
composite = (weights["climat"] * climat +
             weights["sol"] * sol +
             weights["topo"] * topo)

# --- Renormalisation sur [0,1] en ignorant les NaN ---
min_val = np.nanmin(composite)
max_val = np.nanmax(composite)
composite_norm = (composite - min_val) / (max_val - min_val)

# --- Replacer les NaN par le no-data du profil ---
composite_norm[np.isnan(composite_norm)] = profile.get('nodata', np.nan)

# --- Sauvegarde ---
profile.update(dtype=rasterio.float32)

output_path = "../data/processed/Indices finaux/Composites/indice_eq_ref91.tif"
with rasterio.open(output_path, 'w', **profile) as dst:
    dst.write(composite_norm.astype(rasterio.float32), 1)

print(f"Indice composite calculé et sauvegardé : {output_path}")


In [ ]:
normalize_with_clip(
    input_path="../data/processed/Indices finaux/Composites/indice_eq_ref91.tif",
    output_path="../data/processed/Indices finaux/Composites/indice_eq_ref91_norm.tif",
    pmin=2.0,
    pmax=98.0,
    prelog=False,
    symmetric=False,
    nodata_keep=True,
    overwrite=True
)